# RSNA Knee: image-model training (Kaggle)

Trains the same 2.5D multi-slot MIL model as `src/imaging/train.py`.
- **Step 1** rebuilds the study index from DICOM headers: slot choice, knee side (tag first, else coronal/axial geometry), and the 2.5D step.
- **Step 2** caches only the selected series with the exact training preprocessing (`RUN["preproc"]`: v2 224 px or v6 256 px at 0.6 mm/px).
- **Step 3** trains with pseudo-labelled validation (fold 0) and saves sealed gold predictions. Each epoch writes a checkpoint, so the run is resumable.
- **Step 4** packages artefacts into `/kaggle/working/rsna_artifacts` plus zip archives.

Attach the competition data and a **private** dataset holding `training_labels_v3.csv`; never publish it. Internet is only needed if
timm pretrained weights are not attached. Tested locally: construction, forward, loss, checkpoint save/load, packaging.
Untested: the full Kaggle run (no Kaggle credentials on the dev machine).

## 1. Environment and dependencies

In [ ]:
import glob, os, subprocess, sys, json, time, shutil
def find_dir(pattern_files, roots=("/kaggle/input",)):
    for root in roots:
        for p in glob.glob(os.path.join(root, "**", pattern_files[0]), recursive=True):
            d = os.path.dirname(p)
            if all(os.path.exists(os.path.join(d, f)) for f in pattern_files):
                return d
    return None
def pip_offline(pkgs):
    whl_dirs = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/*.whl", recursive=True)})
    if not whl_dirs:
        print("no wheel directories attached"); return False
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "-q"] + sum([["--find-links", d] for d in whl_dirs], []) + pkgs
    print("installing", pkgs, "from", whl_dirs); return subprocess.run(cmd).returncode == 0

In [ ]:
import torch
print('torch', torch.__version__, '| GPUs', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.is_available()

In [ ]:
# Offline dependencies. Install BEFORE pydicom is first imported: pixel-decoder plugins are discovered
# at import time. Wheels in the attached dataset cover CPython 3.10-3.14 (Kaggle image: Python 3.13).
from importlib.metadata import version, PackageNotFoundError
def _ver(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None
print("python", sys.version.split()[0], "| pydicom", _ver("pydicom"), "| pylibjpeg", _ver("pylibjpeg"),
      "| libjpeg", _ver("pylibjpeg-libjpeg"), "| openjpeg", _ver("pylibjpeg-openjpeg"), "| gdcm", _ver("python-gdcm"), "| timm", _ver("timm"))
assert "pydicom" not in sys.modules, "pydicom was imported before the dependency cell: restart the kernel and run from the top"
PYDICOM_VERSION, TIMM_VERSION = "3.0.2", "1.0.30"          # versions used locally (timm: training)
if _ver("pydicom") is None or int(_ver("pydicom").split(".")[0]) < 3:
    pip_offline([f"pydicom=={PYDICOM_VERSION}", "--no-deps"])
missing = [p for p in ("pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg") if _ver(p) is None]
if missing:
    pip_offline(missing)
if _ver("timm") != TIMM_VERSION:
    print(f"timm {_ver('timm')} != {TIMM_VERSION}: installing the bundled wheel")
    pip_offline([f"timm=={TIMM_VERSION}", "--no-deps"])
import pydicom, timm
from pydicom.pixels import get_decoder
from pydicom.uid import JPEGLosslessSV1, JPEGLossless, JPEG2000Lossless, JPEG2000
ok = {u.name: get_decoder(u).is_available for u in (JPEGLosslessSV1, JPEGLossless, JPEG2000Lossless, JPEG2000)}
print("pydicom", pydicom.__version__, "| timm", timm.__version__, "| decoders", ok)
assert all(ok.values()), "JPEG-Lossless / JPEG-2000 decoders unavailable: attach the dataset with the wheels (internet is off)"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} loaded; restart the kernel after installation"

## 2. Configuration

In [ ]:
# ---------------- CONFIG ----------------
COMP_DIR = find_dir(["train.csv", "train_series.csv"]) or "/kaggle/input/rsna-knee-abnormality-detection"
LABELS_CSV = (glob.glob("/kaggle/input/**/training_labels_v3.csv", recursive=True) or [None])[0]   # private dataset you upload
WORK = "/kaggle/working"
RUN = dict(run_id="kaggle_final", backbone="convnext_tiny.fb_in22k_ft_in1k", slots="0,1,2", K=8, agg="attn12",
           preproc="v2",                             # "v2" = 224 px squash, "v6" = 0.6 mm/px aspect-preserving 256 px crop
           epochs=8, bs=2, accum=2, lr_bb=1e-4, lr_head=5e-4, gold_weight=0.0, train_all_folds=False, val_fold=0,
           seed=42, workers=4, time_limit_min=0,     # time_limit_min>0: stop cleanly and resume in a new session
           extra_args=["--sag-pos-fix"])             # final recipe (F1-F3); add other train.py flags here
# Final models F1-F3 = these defaults with train_all_folds=True and seeds 42 / 7 / 11 (one run each).
CACHE = (f"/kaggle/tmp/knee_cache_{RUN['preproc']}" if os.path.isdir("/kaggle/tmp") else f"/tmp/knee_cache_{RUN['preproc']}")
SMOKE = False   # True: tiny run (cache 40 studies, 1 epoch) to check the notebook end to end
print(COMP_DIR, LABELS_CSV, CACHE)
assert LABELS_CSV, "attach a private dataset containing training_labels_v3.csv"

## 3. Code (generated from `src/imaging/*.py` and `src/labeling/metrics.py`)

In [ ]:
import os, sys, math, time, json, argparse
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
"""Build the per-study index used by the image models (train) and the same logic for test.

For every study it decides:
  * which series fills each slot  (plane x {fluid-sensitive, non-fluid-sensitive}),
  * the knee side used for orientation normalisation (+ provenance and conflict flags),
  * the 2.5D neighbour offset per series from its physical slice spacing,
  * the CV fold (train studies only).

Series *metadata* (plane, Fluid_Sensitive flag, slice count, spacing, geometry) is used only
to choose WHICH images are read. It is never fed to the classifier as a feature.

Laterality policy (per study):
  side = majority DICOM Laterality tag if any series carries one, else
         majority geometric side over series whose volume centre is >= 20 mm from x=0
         (sign of patient-x: + -> left knee), else majority over all series.
  Flags: 'conflict' if a confident geometric side disagrees with the tag side;
         'ambiguous' if no series is >= 20 mm off the midline.
"""
import argparse
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

PLANES = ["Sagittal", "Coronal", "Axial"]
SLOTS = [(p, fs) for fs in (1, 0) for p in PLANES]          # 0..2 = FS sag/cor/ax, 3..5 = non-FS
SLOT_NAMES = [f"{p[:3].lower()}_{'fs' if fs else 'nfs'}" for p, fs in SLOTS]
TARGET_STEP_MM = 3.5      # physical distance between the 3 slices of a 2.5D window
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]


def choose_slots(ser):
    """ser: DataFrame of one study's series with columns SeriesInstanceUID, Anatomical_Plane,
    Fluid_Sensitive, n_slices. Returns dict slot_idx -> SeriesInstanceUID (or None) and the
    candidate lists (for optional random choice during training)."""
    out, cands = {}, {}
    for si, (plane, fs) in enumerate(SLOTS):
        c = ser[(ser.Anatomical_Plane == plane) & (ser.Fluid_Sensitive == fs) & (ser.n_slices >= 3)]
        c = c.sort_values(["n_slices", "SeriesInstanceUID"], ascending=[False, True])
        cands[si] = list(c.SeriesInstanceUID)
        out[si] = cands[si][0] if cands[si] else None
    return out, cands


def study_side(lat_rows):
    """lat_rows: rows of laterality_by_series for one study."""
    tags = lat_rows.tag_side.dropna()
    tags = tags[tags.isin(["L", "R"])]
    # Geometry only from coronal/axial series: their in-plane centre-x is exact for every slice,
    # whereas sagittal slices are spread along x (v2 index fix; see image_pipeline_audit.md).
    ca = lat_rows[lat_rows.plane.isin(["Coronal", "Axial"])]
    ca = ca if len(ca) else lat_rows
    conf = ca[ca.center_x_mm.abs() >= 20]
    geo_src = conf if len(conf) else ca
    geo = geo_src.geo_side.value_counts().idxmax()
    ambiguous = len(conf) == 0
    if len(tags):
        tag = tags.value_counts().idxmax()
        conflict = (not ambiguous) and (conf.geo_side.value_counts().idxmax() != tag)
        return tag, "tag", bool(conflict), ambiguous
    return geo, "geometry", False, ambiguous


def build(series_df, headers_df, lat_df):
    s = series_df.merge(headers_df, on="SeriesInstanceUID", how="left")
    rows = []
    for uid, ser in s.groupby("StudyInstanceUID"):
        slots, cands = choose_slots(ser)
        side, src, conflict, amb = study_side(lat_df[lat_df.StudyInstanceUID == uid])
        r = {"StudyInstanceUID": uid, "side": side, "side_source": src, "side_conflict": conflict,
             "side_ambiguous": amb}
        for si, name in enumerate(SLOT_NAMES):
            sid = slots[si]
            r[f"{name}_uid"] = sid
            r[f"{name}_n_cands"] = len(cands[si])
            r[f"{name}_cands"] = "|".join(cands[si])
            if sid is not None:
                h = ser[ser.SeriesInstanceUID == sid].iloc[0]
                sp = h.spacing_mm if np.isfinite(h.spacing_mm) and h.spacing_mm > 0 else 3.5
                r[f"{name}_n"] = int(h.n_slices)
                r[f"{name}_spacing"] = float(sp)
                r[f"{name}_step"] = int(max(1, round(TARGET_STEP_MM / sp)))
            else:
                r[f"{name}_n"], r[f"{name}_spacing"], r[f"{name}_step"] = 0, np.nan, 0
        rows.append(r)
    return pd.DataFrame(rows)


def add_folds(idx, labels, lang, dup_groups, n_folds=5, seed=42):
    """Stratify non-gold studies by language x number of report-positive findings.
    Gold studies get fold -1 (never in a training/validation fold). Duplicate studies share a fold."""
    lab = labels.set_index("StudyInstanceUID")
    gold = set(lab.index[lab.label_source == "gold"])
    ng = idx[~idx.StudyInstanceUID.isin(gold)].copy()
    burden = (lab.loc[ng.StudyInstanceUID, TARGETS] >= 0.5).sum(axis=1).clip(upper=6).astype(str).values
    key = (lang.reindex(ng.StudyInstanceUID).fillna("NA").values + "_" + burden)
    vc = pd.Series(key).value_counts()
    key = np.where(pd.Series(key).map(vc).values < n_folds, "rare", key)
    folds = np.full(len(ng), -1)
    for f, (_, va) in enumerate(StratifiedKFold(n_folds, shuffle=True, random_state=seed).split(ng, key)):
        folds[va] = f
    ng["fold"] = folds
    fmap = dict(zip(ng.StudyInstanceUID, ng.fold))
    for grp in dup_groups:                       # force duplicates into the first member's fold
        f0 = fmap.get(grp[0])
        for u in grp[1:]:
            if u in fmap and f0 is not None:
                fmap[u] = f0
    idx["fold"] = idx.StudyInstanceUID.map(fmap).fillna(-1).astype(int)
    idx["is_gold"] = idx.StudyInstanceUID.isin(gold)
    return idx


def headers_table(series_headers_csv):
    h = pd.read_csv(series_headers_csv, dtype=str)
    out = pd.DataFrame({"SeriesInstanceUID": h.SeriesInstanceUID_dir,
                        "n_slices": pd.to_numeric(h.n_files),
                        "spacing_mm": pd.to_numeric(h.SpacingBetweenSlices, errors="coerce").fillna(
                            pd.to_numeric(h.SliceThickness, errors="coerce"))})
    return out

In [ ]:
"""Evaluation utilities shared by the label audit, the new label pipeline and the
downstream experiments.

All functions take plain numpy arrays. ROC-AUC is reported as NaN (never imputed)
when the evaluation subset contains a single class. Confidence intervals come
from a study-level bootstrap; for macro metrics the same resampled studies are
used for every class, so the interval reflects the joint uncertainty.
"""
import numpy as np
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]


def roc_auc_score(y, s):
    """Mann-Whitney AUC with average ranks for ties (same value as sklearn)."""
    y = np.asarray(y)
    r = rankdata(s)
    n_pos = int((y == 1).sum())
    n_neg = len(y) - n_pos
    return (r[y == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


def safe_auc(y, s):
    y = np.asarray(y, dtype=float)
    s = np.asarray(s, dtype=float)
    m = ~(np.isnan(y) | np.isnan(s))
    y, s = y[m], s[m]
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def ece(y, p, n_bins=5):
    """Expected calibration error with equal-width bins (few bins: n is small)."""
    y = np.asarray(y, dtype=float)
    p = np.clip(np.asarray(p, dtype=float), 0, 1)
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, bins[1:-1]), 0, n_bins - 1)
    total = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            total += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(total)


def binary_report(y, s, threshold=0.5, n_boot=2000, seed=0):
    """Metrics for one target. `s` may be a probability or any monotone score;
    calibration metrics are only meaningful when it is a probability in [0,1]."""
    y = np.asarray(y, dtype=float)
    s = np.asarray(s, dtype=float)
    m = ~(np.isnan(y) | np.isnan(s))
    y, s = y[m], s[m]
    n = len(y)
    pos = int(y.sum())
    pred = (s >= threshold).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else np.nan
    rec = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    f1 = 2 * prec * rec / (prec + rec) if (tp and not np.isnan(prec) and not np.isnan(rec)) else (0.0 if tp + fp + fn else np.nan)
    auc = safe_auc(y, s)
    lo = hi = np.nan
    if not np.isnan(auc) and n_boot:
        rng = np.random.default_rng(seed)
        pos_i, neg_i = np.where(y == 1)[0], np.where(y == 0)[0]
        boots = []
        for _ in range(n_boot):  # stratified: keep class counts fixed
            bi = np.concatenate([rng.choice(pos_i, len(pos_i)), rng.choice(neg_i, len(neg_i))])
            boots.append(roc_auc_score(y[bi], s[bi]))
        lo, hi = np.percentile(boots, [2.5, 97.5])
    is_prob = bool(s.min() >= 0 and s.max() <= 1)
    pc = np.clip(s, 1e-4, 1 - 1e-4)
    return {
        "n": n, "n_pos": pos, "n_neg": n - pos,
        "auc": auc, "auc_lo": lo, "auc_hi": hi,
        "ap": float(average_precision_score(y, s)) if 0 < pos < n else np.nan,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "precision": prec, "recall": rec, "specificity": spec, "f1": f1,
        "balanced_acc": np.nanmean([rec, spec]) if not (np.isnan(rec) and np.isnan(spec)) else np.nan,
        "accuracy": (tp + tn) / n if n else np.nan,
        "brier": float(np.mean((s - y) ** 2)) if is_prob else np.nan,
        "log_loss": float(-np.mean(y * np.log(pc) + (1 - y) * np.log(1 - pc))) if is_prob else np.nan,
        "ece5": ece(y, s) if is_prob else np.nan,
        "mean_score_pos": float(s[y == 1].mean()) if pos else np.nan,
        "mean_score_neg": float(s[y == 0].mean()) if n - pos else np.nan,
    }


def macro_auc_ci(Y, S, n_boot=2000, seed=0):
    """Macro AUC over columns with both classes, with a joint study bootstrap CI.

    Y, S: (n_studies, n_targets) arrays; NaN in S marks a missing score and is
    excluded per column.
    """
    Y = np.asarray(Y, dtype=float)
    S = np.asarray(S, dtype=float)
    point = np.nanmean([safe_auc(Y[:, j], S[:, j]) for j in range(Y.shape[1])])
    rng = np.random.default_rng(seed)
    n = Y.shape[0]
    boots = []
    for _ in range(n_boot):
        bi = rng.integers(0, n, n)
        vals = [safe_auc(Y[bi, j], S[bi, j]) for j in range(Y.shape[1])]
        vals = [v for v in vals if not np.isnan(v)]
        if vals:
            boots.append(np.mean(vals))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return float(point), float(lo), float(hi)


def paired_macro_auc_diff(Y, S1, S2, n_boot=2000, seed=0):
    """Bootstrap distribution of macro-AUC(S1) - macro-AUC(S2) on the same studies.
    Returns (point difference, 2.5%, 97.5%, fraction of bootstrap draws <= 0)."""
    Y = np.asarray(Y, dtype=float)
    def mac(S, idx):
        v = [safe_auc(Y[idx, j], S[idx, j]) for j in range(Y.shape[1])]
        v = [x for x in v if not np.isnan(x)]
        return np.mean(v) if v else np.nan
    n = Y.shape[0]
    full = np.arange(n)
    point = mac(S1, full) - mac(S2, full)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        bi = rng.integers(0, n, n)
        d = mac(S1, bi) - mac(S2, bi)
        if not np.isnan(d):
            diffs.append(d)
    diffs = np.array(diffs)
    return float(point), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5)), float((diffs <= 0).mean())

In [ ]:
"""DICOM -> model-input preprocessing for inference (and for re-building caches).

Reproduces the v2 training cache exactly (verified bit-exact on the local test studies by
src/imaging/check_infer_exactness.py):
  order slices by ImagePositionPatient . slice-normal (fallback InstanceNumber, then name)
  -> RescaleSlope/Intercept, MONOCHROME1 inversion, middle frame of multi-frame files
  -> per-series 1st/99th percentile window -> uint8 -> PIL bilinear resize to 224x224.
Differences from v2, on purpose: v2 silently skipped ANY slice that failed to decode. Here only one
undecodable slice at either end of the slice order is dropped, and it is reported; any other failure
raises DecodeError (see decode_series). An unreadable series is reported, never replaced by zeros.

The study index for unseen studies (slots, knee side, 2.5D step) is rebuilt from the DICOM
headers with the same rules as src/imaging/study_index.py.
"""
import os

import numpy as np
import pandas as pd
import pydicom
from PIL import Image




class DecodeError(RuntimeError):
    pass


def process_pool(workers):
    """Process pool that also works for functions defined inside a notebook: 'fork' where available
    (Linux/Kaggle; Python 3.14 changed the POSIX default to 'forkserver', which cannot pickle them)."""
    import multiprocessing as mp
    from concurrent.futures import ProcessPoolExecutor
    if "fork" in mp.get_all_start_methods():
        return ProcessPoolExecutor(workers, mp_context=mp.get_context("fork"))
    return ProcessPoolExecutor(workers)


def order_series_files(series_dir):
    fnames = [f for f in os.listdir(series_dir) if f.lower().endswith(".dcm")]
    if not fnames:
        raise DecodeError(f"no .dcm files in {series_dir}")
    entries = []
    for f in fnames:
        try:
            ds = pydicom.dcmread(os.path.join(series_dir, f))
        except Exception as ex:
            raise DecodeError(f"unreadable header {series_dir}/{f}: {type(ex).__name__}: {ex}")
        entries.append([None, f, ds])
    have_geo = True
    for e in entries:
        try:
            iop = np.array(e[2].ImageOrientationPatient, dtype=np.float64)
            ipp = np.array(e[2].ImagePositionPatient, dtype=np.float64)
            e[0] = float(np.dot(np.cross(iop[:3], iop[3:]), ipp))
        except Exception:
            have_geo = False
            break
    if have_geo and len({round(e[0], 3) for e in entries}) > 1:
        entries.sort(key=lambda e: e[0])
        return [(e[1], e[2]) for e in entries]
    try:
        nums = [int(getattr(e[2], "InstanceNumber", -1)) for e in entries]
        if len(set(nums)) == len(entries) and all(n >= 0 for n in nums):
            order = sorted(range(len(entries)), key=lambda i: nums[i])
            return [(entries[i][1], entries[i][2]) for i in order]
    except Exception:
        pass
    entries.sort(key=lambda e: e[1])
    return [(e[1], e[2]) for e in entries]


def decode_slice(ds):
    arr = ds.pixel_array.astype(np.float32)        # raises if no decoder is available
    if arr.ndim == 3:
        arr = arr[arr.shape[0] // 2]
    arr = arr * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def decode_series(ordered, series_dir):
    """Decode every slice of an ordered series -> (list of float arrays, list of dropped file names).

    Exactly ONE undecodable slice at either END of the slice order, in a series of >= 10 slices, is
    dropped and reported (seen twice in the 24k training series: a truncated first/last file; the
    legacy v2 cache dropped the same two slices). Any other decode failure raises DecodeError, so the
    caller falls back to another series or masks the slot; nothing is ever zero-filled."""
    slices, errs = [], []
    for i, (f, ds) in enumerate(ordered):
        try:
            slices.append(decode_slice(ds))
        except Exception as ex:
            errs.append((i, f, ds, ex))
    if not errs:
        return slices, []
    i, f, ds, ex = errs[0]
    if len(errs) == 1 and i in (0, len(ordered) - 1) and len(ordered) >= 10:
        return slices, [f]
    raise DecodeError(f"pixel decode failed {series_dir}/{f} (transfer syntax {getattr(ds.file_meta, 'TransferSyntaxUID', '?')}): "
                      f"{type(ex).__name__}: {ex} [{len(errs)} of {len(ordered)} slices failed]")


def preprocess_series(series_dir, img_size=224):
    """-> (uint8 [T, img_size, img_size], geometry dict). Raises DecodeError on failure (see decode_series)."""
    ordered = order_series_files(series_dir)
    slices, dropped = decode_series(ordered, series_dir)
    shapes = {s.shape for s in slices}
    if len(shapes) != 1:
        raise DecodeError(f"inconsistent slice shapes {shapes} in {series_dir}")
    stack = np.stack(slices, 0)
    lo, hi = np.percentile(stack, [1.0, 99.0])
    hi = max(hi, lo + 1.0)
    stack = (np.clip((stack - lo) / (hi - lo), 0, 1) * 255.0).astype(np.uint8)
    out = np.stack([np.array(Image.fromarray(s).resize((img_size, img_size), Image.BILINEAR), dtype=np.uint8)
                    for s in stack])
    if not np.isfinite(out).all() or out.max() == out.min():
        raise DecodeError(f"degenerate (constant) series {series_dir}")
    return out, {**series_geometry(ordered), "dropped_slices": dropped}


def preprocess_series_v6(series_dir, out_size=256, target_mm=0.6):
    """v6: same decode/order/percentile as v2, then ASPECT-PRESERVING physical resampling to
    `target_mm` per pixel (PIL bilinear) and a centred out_size x out_size crop/zero-pad
    (256 x 0.6 mm = 153.6 mm field of view). -> (uint8 [T, out, out], geometry dict)."""
    ordered = order_series_files(series_dir)
    slices, dropped = decode_series(ordered, series_dir)
    shapes = {s.shape for s in slices}
    if len(shapes) != 1:
        raise DecodeError(f"inconsistent slice shapes {shapes} in {series_dir}")
    stack = np.stack(slices, 0)
    lo, hi = np.percentile(stack, [1.0, 99.0])
    hi = max(hi, lo + 1.0)
    stack = (np.clip((stack - lo) / (hi - lo), 0, 1) * 255.0).astype(np.uint8)
    ds0 = ordered[0][1]
    try:
        ps_r, ps_c = [float(v) for v in ds0.PixelSpacing]
    except Exception:
        ps_r = ps_c = 160.0 / max(stack.shape[1:])            # assume a 160 mm FOV if spacing is missing
    H, W = stack.shape[1:]
    nh, nw = max(1, int(round(H * ps_r / target_mm))), max(1, int(round(W * ps_c / target_mm)))
    out = np.zeros((len(stack), out_size, out_size), np.uint8)
    h, w = min(out_size, nh), min(out_size, nw)                 # copied extent
    sy, sx = max(0, (nh - out_size) // 2), max(0, (nw - out_size) // 2)   # centred crop in the resized image
    dy, dx = max(0, (out_size - nh) // 2), max(0, (out_size - nw) // 2)   # centred placement (zero pad)
    for i, s in enumerate(stack):
        r = np.array(Image.fromarray(s).resize((nw, nh), Image.BILINEAR), dtype=np.uint8)
        out[i, dy:dy + h, dx:dx + w] = r[sy:sy + h, sx:sx + w]
    if out.max() == out.min():
        raise DecodeError(f"degenerate (constant) series {series_dir}")
    return out, {**series_geometry(ordered), "dropped_slices": dropped}


def series_geometry(ordered):
    ds0 = ordered[0][1]
    geo = {"n_slices": len(ordered), "tag_side": "", "center_x_mm": np.nan, "spacing_mm": np.nan}
    lat = str(getattr(ds0, "Laterality", "") or "").strip().upper()[:1]
    geo["tag_side"] = lat if lat in ("L", "R") else ""
    try:
        iop = np.array(ds0.ImageOrientationPatient, dtype=float)
        ps = [float(v) for v in ds0.PixelSpacing]
        ipp = np.array(ds0.ImagePositionPatient, dtype=float)
        c = ipp + iop[:3] * ps[1] * int(ds0.Columns) / 2 + iop[3:] * ps[0] * int(ds0.Rows) / 2
        geo["center_x_mm"] = float(c[0])
    except Exception:
        pass
    # Same source priority as the training index (study_index.headers_table):
    # SpacingBetweenSlices -> SliceThickness -> measured position spacing -> 3.5 mm.
    for tag in ("SpacingBetweenSlices", "SliceThickness"):
        try:
            v = float(getattr(ds0, tag))
            if np.isfinite(v) and v > 0:
                geo["spacing_mm"] = v
                break
        except Exception:
            continue
    if not np.isfinite(geo["spacing_mm"]):
        try:
            iop = np.array(ds0.ImageOrientationPatient, dtype=float)
            n = np.cross(iop[:3], iop[3:])
            pos = [float(np.dot(n, np.array(d.ImagePositionPatient, dtype=float))) for _, d in ordered]
            geo["spacing_mm"] = float(np.median(np.abs(np.diff(pos)))) if len(pos) > 1 else 3.5
        except Exception:
            geo["spacing_mm"] = 3.5
    return geo


def series_header_geometry(series_dir):
    """Header-only pass (no pixel decoding): slice count, tag side, centre-x, spacing."""
    fnames = [f for f in os.listdir(series_dir) if f.lower().endswith(".dcm")]
    if not fnames:
        raise DecodeError(f"no .dcm files in {series_dir}")
    entries = []
    for f in fnames:
        try:
            ds = pydicom.dcmread(os.path.join(series_dir, f), stop_before_pixels=True)
        except Exception as ex:
            raise DecodeError(f"unreadable header {series_dir}/{f}: {type(ex).__name__}: {ex}")
        try:
            iop = np.array(ds.ImageOrientationPatient, dtype=float)
            key = float(np.dot(np.cross(iop[:3], iop[3:]), np.array(ds.ImagePositionPatient, dtype=float)))
        except Exception:
            key = 0.0
        entries.append((key, f, ds))
    entries.sort(key=lambda e: e[0])
    return series_geometry([(f, ds) for _, f, ds in entries])


def study_side_from_geo(geos):
    """geos: list of geometry dicts (with 'plane') for one study's series; same policy as
    study_index.study_side: tag majority if any, else coronal/axial geometric majority."""
    tags = [g["tag_side"] for g in geos if g["tag_side"] in ("L", "R")]
    ca = [g for g in geos if g.get("plane") in ("Coronal", "Axial") and np.isfinite(g["center_x_mm"])]
    ca = ca if ca else [g for g in geos if np.isfinite(g["center_x_mm"])]
    conf = [g for g in ca if abs(g["center_x_mm"]) >= 20]
    src = conf if conf else ca
    geo_votes = ["L" if g["center_x_mm"] > 0 else "R" for g in src]
    geo = max(set(geo_votes), key=geo_votes.count) if geo_votes else "L"
    if tags:
        tag = max(set(tags), key=tags.count)
        conflict = bool(conf) and (max(set(["L" if g["center_x_mm"] > 0 else "R" for g in conf]),
                                       key=["L" if g["center_x_mm"] > 0 else "R" for g in conf].count) != tag)
        return tag, "tag", conflict, not conf
    return geo, "geometry", False, not conf


def build_index_row(study_uid, series_meta, geos):
    """series_meta: DataFrame (SeriesInstanceUID, Anatomical_Plane, Fluid_Sensitive) for one study;
    geos: dict series_uid -> geometry dict (from preprocess_series)."""
    ser = series_meta.copy()
    ser["n_slices"] = ser.SeriesInstanceUID.map(lambda s: geos[s]["n_slices"] if s in geos else 0)
    slots, cands = choose_slots(ser)
    plane_of = dict(zip(ser.SeriesInstanceUID, ser.Anatomical_Plane))
    side, src, conflict, amb = study_side_from_geo([{**geos[s], "plane": plane_of[s]} for s in ser.SeriesInstanceUID if s in geos])
    r = {"StudyInstanceUID": study_uid, "side": side, "side_source": src, "side_conflict": conflict, "side_ambiguous": amb}
    for si, name in enumerate(SLOT_NAMES):
        sid = slots[si]
        r[f"{name}_uid"], r[f"{name}_n_cands"], r[f"{name}_cands"] = sid, len(cands[si]), "|".join(cands[si])
        if sid is not None:
            sp = geos[sid]["spacing_mm"]
            r[f"{name}_n"], r[f"{name}_spacing"] = geos[sid]["n_slices"], sp
            r[f"{name}_step"] = int(max(1, round(TARGET_STEP_MM / sp)))
        else:
            r[f"{name}_n"], r[f"{name}_spacing"], r[f"{name}_step"] = 0, np.nan, 0
    return r

In [ ]:
"""Study-level dataset for 2.5D multi-slot MRI models.

Reads per-series uint8 volumes (T, H, W) from a preprocessing cache (default: the verified
v2 cache: per-series 1-99 percentile window -> uint8 -> 224x224 bilinear, slices sorted
along the slice normal). Orientation normalisation (applied here, at load time):

  * coronal / axial, right knee  -> mirror columns  (lateral is then always image-right)
  * sagittal, right knee         -> reverse slice order (slice 0 is then always lateral)

The sagittal rule follows from the data's uniform orientation (rows +y, cols -z, slice
normal -x for every sagittal series): ascending normal projection runs from patient-left
to patient-right, i.e. lateral->medial for a left knee and medial->lateral for a right knee.

Each slot yields K windows of 3 slices [c-step, c, c+step] (step from physical spacing, about
3.5 mm). Missing slots are zeros with mask 0; the model must ignore them.
"""
import os

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset




def window_centres(T, K, lo, hi, train, rng):
    a, b = lo * (T - 1), hi * (T - 1)
    if K == 1:
        c = np.array([(a + b) / 2])
    elif train:
        edges = np.linspace(a, b, K + 1)
        c = rng.uniform(edges[:-1], edges[1:])
    else:
        c = np.linspace(a, b, K)
    return np.clip(np.round(c).astype(int), 0, T - 1)


class KneeStudyDataset(Dataset):
    def __init__(self, index_df, cache_dir, slots=(0, 1, 2), K=8, train=False, labels=None,
                 weights=None, lo=0.04, hi=0.96, random_series=False, spacing_map=None, seed=0,
                 orient=True, strict=True, fallback=True, sag_pos_fix=False):
        self.idx = index_df.reset_index(drop=True)
        self.cache_dir = cache_dir
        self.slots = list(slots)
        self.K, self.train, self.lo, self.hi = K, train, lo, hi
        self.labels = labels            # (N, 12) float array aligned with index_df, or None
        self.weights = weights          # (N,) float array or None
        self.random_series = random_series
        self.spacing_map = spacing_map or {}
        self.seed = seed
        self.orient = orient
        self.strict = strict            # missing cache file for a chosen series -> raise, never zero-fill
        self.fallback = fallback        # empty FS slot (0-2) -> use the same plane's non-FS series (slot+3)
        # Slice-position feature for RIGHT-knee sagittal series. The mirrored window indices already put
        # window k at anatomical position p = c/(T-1) (0 = lateral) for both sides, so the correct feature
        # is p. The original code passed 1 - p (found 2026-10-17 by attention_map.py); False reproduces it
        # for checkpoints trained that way (args without "sag_pos_fix").
        self.sag_pos_fix = sag_pos_fix

    def __len__(self):
        return len(self.idx)

    def _load(self, uid):
        p = os.path.join(self.cache_dir, f"{uid}.npy")
        if not os.path.exists(p):
            if self.strict:
                raise FileNotFoundError(f"cache missing for series {uid}")
            return None
        return np.load(p, mmap_mode="r")

    def __getitem__(self, i):
        row = self.idx.iloc[i]
        rng = np.random.default_rng((self.seed * 1_000_003 + i * 7919 + int(torch.randint(0, 1 << 30, (1,)))) if self.train else i)
        S = len(self.slots)
        x, mask, pos = None, np.zeros(S, np.float32), np.zeros((S, self.K), np.float32)
        right = (row["side"] == "R") and self.orient
        for j, si in enumerate(self.slots):
            name = SLOT_NAMES[si]
            uid = row[f"{name}_uid"]
            if not isinstance(uid, str) and self.fallback and si < 3 and (si + 3) not in self.slots:
                name = SLOT_NAMES[si + 3]          # same plane, non-fluid-sensitive series
                uid = row[f"{name}_uid"]
            if not isinstance(uid, str):
                continue
            if self.train and self.random_series:
                cands = str(row[f"{name}_cands"]).split("|")
                uid = cands[rng.integers(len(cands))]
            step = int(row[f"{name}_step"]) if uid == row[f"{name}_uid"] else \
                max(1, int(round(3.5 / self.spacing_map.get(uid, 3.5))))
            arr = self._load(uid)
            if arr is None:
                continue
            T, H, W = arr.shape
            if x is None:
                x = np.zeros((S, self.K, 3, H, W), np.uint8)
            plane = SLOTS[si][0]
            cs = window_centres(T, self.K, self.lo, self.hi, self.train, rng)
            for k, c in enumerate(cs):
                ids = np.clip(np.array([c - step, c, c + step]), 0, T - 1)
                if right and plane == "Sagittal":
                    ids = (T - 1) - ids[::-1]          # reversed slice order, channel order lateral->medial
                win = np.asarray(arr[ids])
                if right and plane in ("Coronal", "Axial"):
                    win = win[:, :, ::-1]
                x[j, k] = win
                p = c / max(T - 1, 1)
                pos[j, k] = (1 - p) if (right and plane == "Sagittal" and not self.sag_pos_fix) else p
            mask[j] = 1.0
        if x is None:
            raise RuntimeError(f"study {row['StudyInstanceUID']} has no usable series")
        out = {"x": torch.from_numpy(x), "mask": torch.from_numpy(mask), "pos": torch.from_numpy(pos), "i": i}
        if self.labels is not None:
            out["y"] = torch.from_numpy(self.labels[i].astype(np.float32))
        if self.weights is not None:
            out["w"] = torch.tensor(float(self.weights[i]), dtype=torch.float32)
        return out


def gpu_augment(x, mask, rot_deg=10.0, scale=0.10, shift=0.06, gamma=0.20, gain=0.10):
    """x: float [B,S,K,3,H,W] in [0,1]. One random affine + gamma/gain per (study, slot),
    shared by all windows of that series (keeps the 3 slices of a window aligned)."""
    B, S, K, C, H, W = x.shape
    n = B * S
    dev = x.device
    ang = (torch.rand(n, device=dev) * 2 - 1) * np.deg2rad(rot_deg)
    sc = 1 + (torch.rand(n, device=dev) * 2 - 1) * scale
    tx, ty = [(torch.rand(n, device=dev) * 2 - 1) * shift for _ in range(2)]
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    xs = x.reshape(n, K * C, H, W)
    grid = torch.nn.functional.affine_grid(theta, xs.shape, align_corners=False)
    xs = torch.nn.functional.grid_sample(xs, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    g = torch.exp((torch.rand(n, 1, 1, 1, device=dev) * 2 - 1) * gamma)
    a = 1 + (torch.rand(n, 1, 1, 1, device=dev) * 2 - 1) * gain
    xs = (xs.clamp(0, 1) ** g * a).clamp(0, 1)
    return xs.reshape(B, S, K, C, H, W)


def load_labels(label_csv, uids, transform="soft"):
    lab = pd.read_csv(label_csv).set_index("StudyInstanceUID").reindex(uids)
    y = lab[TARGETS].values.astype(np.float32)
    if np.isnan(y).any():
        raise ValueError(f"{int(np.isnan(y).sum())} missing label cells for the requested studies")
    if transform == "hard":
        y = (y >= 0.5).astype(np.float32)
    return y, lab

In [ ]:
"""2.5D multi-slot MIL model: shared pretrained 2D encoder on 3-slice windows, token
embeddings (slot + slice position), and a choice of study-level aggregation.

Aggregators (agg=):
  mean   masked mean over all window tokens         -> linear 12
  max    masked max over all window tokens          -> linear 12
  gated  Ilse et al. 2018 gated attention, 1 query  -> linear 12
  attn12 per-finding attention (12 queries), each finding scored from its own pooled vector
  tfm    2-layer transformer over tokens, then attn12
"""
import math

import timm
import torch
import torch.nn as nn

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


class KneeMIL(nn.Module):
    def __init__(self, backbone="convnext_tiny.fb_in22k_ft_in1k", pretrained=True, n_slots=3, n_out=12,
                 dim=384, agg="attn12", use_pos=True, use_slot=True, drop_path=0.1, grad_ckpt=False):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, drop_path_rate=drop_path)
        if grad_ckpt:
            self.enc.set_grad_checkpointing(True)
        nf = self.enc.num_features
        self.proj = nn.Sequential(nn.Dropout(0.2), nn.Linear(nf, dim), nn.LayerNorm(dim))
        self.slot_emb = nn.Embedding(n_slots, dim) if use_slot else None
        self.pos_mlp = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim)) if use_pos else None
        self.agg, self.n_out = agg, n_out
        if agg in ("mean", "max"):
            self.head = nn.Linear(dim, n_out)
        elif agg == "gated":
            self.att_v = nn.Linear(dim, 128)
            self.att_u = nn.Linear(dim, 128)
            self.att_w = nn.Linear(128, 1)
            self.head = nn.Linear(dim, n_out)
        elif agg in ("attn12", "tfm"):
            if agg == "tfm":
                layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True, norm_first=True)
                self.tfm = nn.TransformerEncoder(layer, 2)
            self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(0.1), nn.Linear(256, n_out))
            self.cls_w = nn.Parameter(torch.randn(n_out, dim) / math.sqrt(dim))
            self.cls_b = nn.Parameter(torch.zeros(n_out))
        else:
            raise ValueError(agg)
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1), persistent=False)

    def encode(self, x01):
        """x01: float [N, 3, H, W] in [0,1] -> [N, nf]"""
        return self.enc((x01 - self.mean) / self.std)

    def forward(self, x01, mask, pos):
        """x01: float [B,S,K,3,H,W] in [0,1]; mask [B,S]; pos [B,S,K] -> logits [B,12]"""
        B, S, K = x01.shape[:3]
        valid = mask > 0
        feats = x01.new_zeros(B, S, K, self.proj[1].out_features)
        xv = x01[valid]                                          # [Nv, K, 3, H, W]
        f = self.encode(xv.flatten(0, 1).contiguous(memory_format=torch.channels_last))
        feats[valid] = self.proj(f.float()).view(xv.shape[0], K, -1).to(feats.dtype)
        tok = feats
        if self.slot_emb is not None:
            tok = tok + self.slot_emb.weight[:S].view(1, S, 1, -1)
        if self.pos_mlp is not None:
            tok = tok + self.pos_mlp(pos.unsqueeze(-1))
        tok = tok.flatten(1, 2)                                  # [B, S*K, D]
        tmask = valid.unsqueeze(-1).expand(B, S, K).flatten(1)   # True = real token
        if self.agg == "mean":
            m = tmask.unsqueeze(-1).float()
            return self.head((tok * m).sum(1) / m.sum(1).clamp(min=1))
        if self.agg == "max":
            return self.head(tok.masked_fill(~tmask.unsqueeze(-1), -1e4).max(1).values)
        if self.agg == "gated":
            a = self.att_w(torch.tanh(self.att_v(tok)) * torch.sigmoid(self.att_u(tok))).squeeze(-1)
            a = torch.softmax(a.masked_fill(~tmask, -1e4), 1)
            return self.head((a.unsqueeze(-1) * tok).sum(1))
        if self.agg == "tfm":
            tok = self.tfm(tok, src_key_padding_mask=~tmask)
        a = self.att(tok).masked_fill(~tmask.unsqueeze(-1), -1e4)  # [B, T, 12]
        a = torch.softmax(a, 1)
        pooled = torch.einsum("btc,btd->bcd", a, tok)               # [B, 12, D]
        return (pooled * self.cls_w).sum(-1) + self.cls_b

In [ ]:
"""Train / validate a 2.5D multi-slot MIL knee model (local or Kaggle).

Validation protocol (see reports/image_model_experiments.md):
  * train on non-gold studies of folds != val_fold (optionally + gold with --gold-weight > 0),
  * select the checkpoint by macro AUC on the PSEUDO-labelled val fold (teacher binarised at 0.5),
  * predict the 58 gold studies at the selected checkpoint and SAVE THEM SEALED (no metric is
    computed here); gold metrics are computed once, later, by evaluate_sealed.py.
"""
import argparse
import json
import math
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F








R = "D:/Coding/kaggle/RSNA"


def get_args(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--run-id", required=True)
    ap.add_argument("--index", default=f"{R}/data/imaging_v5/study_index_train.csv")
    ap.add_argument("--labels", default=f"{R}/data/labels_v3/training_labels_v3.csv")
    ap.add_argument("--cache", default=f"{R}/output/v2/preprocessed")
    ap.add_argument("--preproc", default="v2", choices=["v2", "v6"],
                    help="preprocessing that produced --cache (recorded for inference): v2 = 224 px squash, "
                         "v6 = 0.6 mm/px aspect-preserving 256 px crop")
    ap.add_argument("--out-root", default=f"{R}/output/imaging_v5/runs")
    ap.add_argument("--backbone", default="convnext_tiny.fb_in22k_ft_in1k")
    ap.add_argument("--slots", default="0,1,2")
    ap.add_argument("--K", type=int, default=8)
    ap.add_argument("--agg", default="attn12")
    ap.add_argument("--no-pos", action="store_true")
    ap.add_argument("--epochs", type=int, default=8)
    ap.add_argument("--bs", type=int, default=2)
    ap.add_argument("--accum", type=int, default=2)
    ap.add_argument("--lr-bb", type=float, default=1e-4)
    ap.add_argument("--lr-head", type=float, default=5e-4)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--warmup", type=float, default=0.5, help="warm-up length in epochs")
    ap.add_argument("--ema", type=float, default=0.999)
    ap.add_argument("--drop-path", type=float, default=0.1)
    ap.add_argument("--label-transform", default="soft", choices=["soft", "hard"])
    ap.add_argument("--gold-weight", type=float, default=0.0, help="0 = gold never trained on")
    ap.add_argument("--pos-weight-clip", type=float, default=0.0, help="0 = plain BCE")
    ap.add_argument("--conf-weight", type=float, default=0.0,
                    help="0 = off; else each label's BCE is weighted by |2y-1|^gamma (down-weights uncertain teacher labels)")
    ap.add_argument("--val-fold", type=int, default=0)
    ap.add_argument("--train-all-folds", action="store_true", help="final model: train on every non-gold fold")
    ap.add_argument("--max-train", type=int, default=0)
    ap.add_argument("--random-series", action="store_true")
    ap.add_argument("--no-orient", action="store_true", help="ablation: disable laterality normalisation")
    ap.add_argument("--sag-pos-fix", action="store_true",
                    help="correct slice-position feature for right-knee sagittal series (see data.KneeStudyDataset)")
    ap.add_argument("--lo", type=float, default=0.04)
    ap.add_argument("--hi", type=float, default=0.96)
    ap.add_argument("--no-aug", action="store_true")
    ap.add_argument("--grad-ckpt", action="store_true")
    ap.add_argument("--amp", default="auto", choices=["auto", "bf16", "fp16"],
                    help="autocast dtype; auto = bf16 on Ampere or newer, else fp16 with loss scaling")
    ap.add_argument("--workers", type=int, default=6)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--time-limit-min", type=float, default=0, help="stop cleanly (resumable) after N minutes")
    return ap.parse_args(argv)


def macro_auc(y_bin, p):
    per = {t: safe_auc(y_bin[:, j], p[:, j]) for j, t in enumerate(TARGETS)}
    vals = [v for v in per.values() if not np.isnan(v)]
    return (float(np.mean(vals)) if vals else float("nan")), per


@torch.no_grad()
def predict(model, loader, dev, amp_dt=torch.bfloat16):
    model.eval()
    out = np.zeros((len(loader.dataset), len(TARGETS)), np.float32)
    for b in loader:
        x = b["x"].to(dev, non_blocking=True).float().div_(255)
        with torch.autocast("cuda", dtype=amp_dt):
            lg = model(x, b["mask"].to(dev), b["pos"].to(dev))
        out[b["i"].numpy()] = torch.sigmoid(lg.float()).cpu().numpy()
    return out


def keep_awake():
    """Prevent Windows from sleeping while this process runs (no system settings are changed)."""
    if os.name == "nt":
        import ctypes
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001)  # ES_CONTINUOUS | ES_SYSTEM_REQUIRED


def main(args):
    keep_awake()
    torch.manual_seed(args.seed); np.random.seed(args.seed)
    run_dir = os.path.join(args.out_root, args.run_id)
    os.makedirs(run_dir, exist_ok=True)
    json.dump(vars(args), open(os.path.join(run_dir, "config.json"), "w"), indent=1)
    dev = "cuda"
    # bf16 on Ampere or newer (the local RTX 4050); fp16 + loss scaling on older GPUs (e.g. Kaggle T4, no native bf16)
    amp_dt = {"bf16": torch.bfloat16, "fp16": torch.float16}.get(
        args.amp, torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_dt == torch.float16)
    idx = pd.read_csv(args.index)
    slots = [int(s) for s in args.slots.split(",")]
    gold = idx[idx.is_gold].reset_index(drop=True)
    if args.train_all_folds:
        tr = idx[idx.fold >= 0]
        va = idx[idx.fold == args.val_fold]        # still reported, but it is in training (not a holdout)
    else:
        tr = idx[(idx.fold >= 0) & (idx.fold != args.val_fold)]
        va = idx[idx.fold == args.val_fold]
    if args.gold_weight > 0:
        tr = pd.concat([tr, gold])
    if args.max_train:
        tr = tr.sample(min(args.max_train, len(tr)), random_state=args.seed)
    tr, va = tr.reset_index(drop=True), va.reset_index(drop=True)
    y_tr, _ = load_labels(args.labels, tr.StudyInstanceUID, args.label_transform)
    w_tr = np.where(tr.is_gold, args.gold_weight, 1.0).astype(np.float32)
    y_va_soft, _ = load_labels(args.labels, va.StudyInstanceUID, "soft")
    y_va_bin = (y_va_soft >= 0.5).astype(np.float32)
    spacing_map = {}                    # only needed for --random-series (non-default candidates)
    hdr_path = f"{R}/reports/eda/dicom/series_headers.csv"
    if args.random_series and os.path.exists(hdr_path):
        hdr = pd.read_csv(hdr_path, dtype=str, usecols=["SeriesInstanceUID_dir", "SpacingBetweenSlices", "SliceThickness"])
        sp = pd.to_numeric(hdr.SpacingBetweenSlices, errors="coerce").fillna(pd.to_numeric(hdr.SliceThickness, errors="coerce"))
        spacing_map = dict(zip(hdr.SeriesInstanceUID_dir, sp.fillna(3.5)))
    orient = not args.no_orient
    ds_tr = KneeStudyDataset(tr, args.cache, slots, args.K, train=True, labels=y_tr, weights=w_tr, lo=args.lo, hi=args.hi,
                             random_series=args.random_series, spacing_map=spacing_map, seed=args.seed, orient=orient,
                             sag_pos_fix=args.sag_pos_fix)
    ds_va = KneeStudyDataset(va, args.cache, slots, args.K, train=False, lo=args.lo, hi=args.hi, orient=orient,
                             sag_pos_fix=args.sag_pos_fix)
    ds_go = KneeStudyDataset(gold, args.cache, slots, args.K, train=False, lo=args.lo, hi=args.hi, orient=orient,
                             sag_pos_fix=args.sag_pos_fix)
    kw = dict(num_workers=args.workers, pin_memory=True, persistent_workers=args.workers > 0,
              prefetch_factor=4 if args.workers > 0 else None)
    dl_tr = torch.utils.data.DataLoader(ds_tr, batch_size=args.bs, shuffle=True, drop_last=True, **kw)
    dl_va = torch.utils.data.DataLoader(ds_va, batch_size=4, shuffle=False, **kw)
    dl_go = torch.utils.data.DataLoader(ds_go, batch_size=4, shuffle=False, num_workers=0)

    model = KneeMIL(args.backbone, True, n_slots=6, agg=args.agg, use_pos=not args.no_pos,
                    drop_path=args.drop_path, grad_ckpt=args.grad_ckpt).to(dev)
    enc_params = list(model.enc.parameters())
    enc_ids = {id(p) for p in enc_params}
    head_params = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": args.lr_bb}, {"params": head_params, "lr": args.lr_head}],
                            weight_decay=args.wd)
    steps_per_epoch = len(dl_tr) // args.accum
    total, warm = steps_per_epoch * args.epochs, int(steps_per_epoch * args.warmup)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, warm)) *
                                             0.5 * (1 + math.cos(math.pi * min(1.0, s / max(1, total)))))
    from timm.utils import ModelEmaV3
    ema = ModelEmaV3(model, decay=args.ema) if args.ema > 0 else None
    pw = None
    if args.pos_weight_clip > 0:
        prev = y_tr.mean(0).clip(1e-3, 1)
        pw = torch.tensor(np.clip((1 - prev) / prev, 1.0, args.pos_weight_clip), device=dev)

    start_epoch, best, hist, step = 0, -1.0, [], 0
    ck_last = os.path.join(run_dir, "ckpt_last.pt")
    if args.resume and os.path.exists(ck_last):
        st = torch.load(ck_last, map_location=dev, weights_only=False)
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"])
        if ema is not None and st.get("ema") is not None:
            ema.module.load_state_dict(st["ema"])
        start_epoch, best, hist, step = st["epoch"] + 1, st["best"], st["hist"], st["step"]
        if st.get("scaler") is not None:
            scaler.load_state_dict(st["scaler"])
        print(f"resumed at epoch {start_epoch}", flush=True)

    t_start = time.time()
    torch.cuda.reset_peak_memory_stats()
    for ep in range(start_epoch, args.epochs):
        model.train()
        t0, run_loss, n_seen = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for it, b in enumerate(dl_tr):
            x = b["x"].to(dev, non_blocking=True).float().div_(255)
            mask, pos = b["mask"].to(dev), b["pos"].to(dev)
            y, w = b["y"].to(dev), b["w"].to(dev)
            if not args.no_aug:
                x = gpu_augment(x, mask)
            with torch.autocast("cuda", dtype=amp_dt):
                lg = model(x, mask, pos)
            loss_el = F.binary_cross_entropy_with_logits(lg.float(), y, pos_weight=pw, reduction="none")
            if args.conf_weight > 0:
                loss_el = loss_el * (2 * y - 1).abs().pow(args.conf_weight)
            loss_el = loss_el.mean(1)
            loss = (loss_el * w).sum() / w.sum().clamp(min=1e-6) / args.accum
            scaler.scale(loss).backward()          # plain backward when the scaler is disabled (bf16)
            run_loss += loss.item() * args.accum * len(y); n_seen += len(y)
            if (it + 1) % args.accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
                if ema is not None:
                    ema.update(model, step=step)
            if it % 200 == 0:
                print(f"ep {ep} it {it}/{len(dl_tr)} loss {run_loss/max(n_seen,1):.4f} "
                      f"{n_seen/(time.time()-t0):.2f} studies/s mem {torch.cuda.max_memory_allocated()/1e9:.2f}GB", flush=True)
        t_train = time.time() - t0
        eval_model = ema.module if ema is not None else model
        t1 = time.time()
        p_va = predict(eval_model, dl_va, dev, amp_dt)
        t_val = time.time() - t1
        va_auc, va_per = macro_auc(y_va_bin, p_va)
        va_bce = float(F.binary_cross_entropy(torch.tensor(p_va).clamp(1e-6, 1 - 1e-6), torch.tensor(y_va_soft)).item())
        row = {"epoch": ep, "train_loss": run_loss / max(n_seen, 1), "val_bce_soft": va_bce, "val_macro_auc": va_auc,
               "train_min": t_train / 60, "val_min": t_val / 60, "train_studies_per_s": n_seen / t_train,
               "peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9, "lr_bb": opt.param_groups[0]["lr"],
               **{f"val_auc_{t}": v for t, v in va_per.items()}}
        hist.append(row)
        pd.DataFrame(hist).to_csv(os.path.join(run_dir, "history.csv"), index=False)
        print(f"== epoch {ep}: train_loss {row['train_loss']:.4f} val_bce {va_bce:.4f} val_macroAUC {va_auc:.4f} "
              f"({t_train/60:.1f}+{t_val/60:.1f} min)", flush=True)
        # --train-all-folds: the val fold is part of training, so selection is meaningless -> keep the LAST epoch
        if va_auc > best or args.train_all_folds:
            best = max(best, va_auc)
            torch.save({"model": eval_model.state_dict(), "args": vars(args), "epoch": ep, "val_macro_auc": va_auc},
                       os.path.join(run_dir, "ckpt_best.pt"))
            pd.DataFrame(p_va, columns=TARGETS).assign(StudyInstanceUID=va.StudyInstanceUID.values).to_csv(
                os.path.join(run_dir, "val_preds_best.csv"), index=False)
            p_go = predict(eval_model, dl_go, dev, amp_dt)       # SEALED: saved, not scored
            np.savez(os.path.join(run_dir, "gold_preds_SEALED.npz"), uids=gold.StudyInstanceUID.values, p=p_go, epoch=ep)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "ema": ema.module.state_dict() if ema is not None else None, "epoch": ep, "best": best,
                    "hist": hist, "step": step, "scaler": scaler.state_dict() if scaler.is_enabled() else None}, ck_last)
        if args.time_limit_min and (time.time() - t_start) / 60 > args.time_limit_min and ep < args.epochs - 1:
            print("time limit reached; stopping (resume with --resume)", flush=True)
            return
    h = pd.DataFrame(hist)
    bi = int(h.val_macro_auc.idxmax())
    summ = {"run_id": args.run_id, "best_epoch": int(h.epoch[bi]), "best_val_macro_auc": float(h.val_macro_auc[bi]),
            "last_val_macro_auc": float(h.val_macro_auc.iloc[-1]), "epochs": args.epochs,
            "total_train_min": float(h.train_min.sum()), "total_val_min": float(h.val_min.sum()),
            "peak_mem_gb": float(h.peak_mem_gb.max()), "n_train": len(tr), "n_val": len(va),
            "val_is_holdout": not args.train_all_folds, "saved_epoch": int(h.epoch.iloc[-1]) if args.train_all_folds else int(h.epoch[bi]),
            "per_class_val_auc_best": {t: float(h[f"val_auc_{t}"][bi]) for t in TARGETS}}
    json.dump(summ, open(os.path.join(run_dir, "summary.json"), "w"), indent=1)
    print("DONE", json.dumps(summ), flush=True)

## 4. Build the train study index from DICOM headers (≈10–20 min on 4 CPUs)

In [ ]:
ts = pd.read_csv(f"{COMP_DIR}/train_series.csv")
if SMOKE:
    ts = ts[ts.StudyInstanceUID.isin(ts.StudyInstanceUID.drop_duplicates().head(40))]
def _geo(a):
    sdir, sid = a
    try:
        return sid, series_header_geometry(sdir), None
    except Exception as ex:
        return sid, None, str(ex)
jobs = [(f"{COMP_DIR}/train_series/{r.StudyInstanceUID}/{r.SeriesInstanceUID}", r.SeriesInstanceUID) for r in ts.itertuples()]
geos, bad = {}, []
with process_pool(os.cpu_count()) as ex:
    for sid, g, e in ex.map(_geo, jobs, chunksize=16):
        (bad.append((sid, e)) if g is None else geos.__setitem__(sid, g))
rows = [build_index_row(st, m[m.SeriesInstanceUID.isin(geos)], geos) for st, m in ts.groupby("StudyInstanceUID")]
index = pd.DataFrame(rows)
labels = pd.read_csv(LABELS_CSV)
labels = labels[labels.StudyInstanceUID.isin(index.StudyInstanceUID)]
index = add_folds(index, labels, pd.Series(dtype=str), dup_groups=[])   # duplicate pair handled in the next cell
print("header failures:", len(bad), "| side sources:", index.side_source.value_counts().to_dict())

Keep the pixel-identical duplicate pair (UIDs ending …23376711 / …31613209) in one fold.

In [ ]:
pair = [u for u in index.StudyInstanceUID if u[-8:] in ("23376711", "31613209")]   # pixel-identical duplicate pair
if len(pair) == 2:
    f0 = index.loc[index.StudyInstanceUID == pair[0], "fold"].item()
    index.loc[index.StudyInstanceUID == pair[1], "fold"] = f0
os.makedirs(f"{WORK}/rsna_artifacts", exist_ok=True)
index.to_csv(f"{WORK}/rsna_artifacts/study_index_train.csv", index=False)

## 5. Cache the selected series (exact training preprocessing, version `RUN['preproc']`)

In [ ]:
os.makedirs(CACHE, exist_ok=True)
slots = [int(s) for s in RUN["slots"].split(",")]
need = set()
for r in index.itertuples():
    for si in slots:
        u = getattr(r, f"{SLOT_NAMES[si]}_uid")
        if not isinstance(u, str) and si < 3:
            u = getattr(r, f"{SLOT_NAMES[si + 3]}_uid")
        if isinstance(u, str):
            need.add((r.StudyInstanceUID, u))
def _cache(a):
    st, sid = a
    p = f"{CACHE}/{sid}.npy"
    if os.path.exists(p):
        return sid, None
    try:
        sdir = f"{COMP_DIR}/train_series/{st}/{sid}"
        arr, _ = preprocess_series_v6(sdir) if RUN["preproc"] == "v6" else preprocess_series(sdir)
        np.save(p + ".tmp.npy", arr); os.replace(p + ".tmp.npy", p); return sid, None
    except Exception as ex:
        return sid, str(ex)
t0 = time.time(); fails = []
with process_pool(os.cpu_count()) as ex:
    for i, (sid, e) in enumerate(ex.map(_cache, sorted(need), chunksize=8)):
        if e: fails.append((sid, e))
        if i % 1000 == 0: print(i, "/", len(need), round(time.time() - t0), "s", flush=True)
print("cached", len(need) - len(fails), "series;", len(fails), "failures")
assert not fails, f"decode failures must be fixed, not skipped: {fails[:3]}"

## 6. Train (resumable: re-running this cell continues from `ckpt_last.pt`)

In [ ]:
argv = ["--run-id", RUN["run_id"], "--index", f"{WORK}/rsna_artifacts/study_index_train.csv", "--labels", LABELS_CSV,
        "--cache", CACHE, "--preproc", RUN["preproc"], "--out-root", f"{WORK}/runs", "--backbone", RUN["backbone"], "--slots", RUN["slots"],
        "--K", str(RUN["K"]), "--agg", RUN["agg"], "--epochs", str(1 if SMOKE else RUN["epochs"]), "--bs", str(RUN["bs"]),
        "--accum", str(RUN["accum"]), "--lr-bb", str(RUN["lr_bb"]), "--lr-head", str(RUN["lr_head"]),
        "--gold-weight", str(RUN["gold_weight"]), "--val-fold", str(RUN["val_fold"]), "--seed", str(RUN["seed"]),
        "--workers", str(RUN["workers"]), "--time-limit-min", str(RUN["time_limit_min"]), "--resume"]
if RUN["train_all_folds"]: argv.append("--train-all-folds")
argv += list(RUN["extra_args"])
if SMOKE: argv += ["--max-train", "24"]
main(get_args(argv))

## 7. Package artefacts (model, configs, metrics, predictions, code) for download

In [ ]:
import zipfile, platform
A = f"{WORK}/rsna_artifacts"; rd = f"{WORK}/runs/{RUN['run_id']}"
for f in ["config.json", "history.csv", "summary.json", "val_preds_best.csv", "gold_preds_SEALED.npz"]:
    if os.path.exists(f"{rd}/{f}"): shutil.copy(f"{rd}/{f}", A)
ck = torch.load(f"{rd}/ckpt_best.pt", map_location="cpu", weights_only=False)
torch.save(ck, f"{WORK}/knee_{RUN['run_id']}.pt")                 # the file the inference notebook loads
json.dump({"targets": TARGETS, "slots": RUN["slots"], "K": RUN["K"], "preprocessing": RUN["preproc"],
           "versions": {"torch": torch.__version__, "timm": timm.__version__, "python": platform.python_version()}},
          open(f"{A}/model_card.json", "w"), indent=1)
with zipfile.ZipFile(f"{WORK}/rsna_model_artifacts_light.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(A): z.write(f"{A}/{f}", f)
print("download from the Output tab:", f"{WORK}/knee_{RUN['run_id']}.pt", "and rsna_model_artifacts_light.zip")